# EDA: Emotional Map of Brighton

**Что этот ноутбук делает, а чего не делает.**

Здесь - только разведка: посмотреть на данные, найти перекосы, проверить
догадки. Ничего из этого не запускается повторно и ничего не попадает
в продакшен.

Всё, что должно выполняться регулярно, вынесено в `src/bem/` и `scripts/`.
Правило, по которому я это разделял:

| В notebook | В .py |
|---|---|
| смотрю на данные один раз | запускаю много раз |
| строю график, чтобы понять | строю график для отчёта |
| проверяю догадку | проверенная догадка стала кодом |
| результат - понимание в голове | результат - файл на диске |

Почему это важно: ноутбук нельзя протестировать, нельзя импортировать
и почти нельзя нормально ревьюить в git - diff показывает мешанину JSON.
Логика, живущая только в ноутбуке, обречена сломаться незаметно.

In [ ]:
import sys
sys.path.insert(0, "../src")

import matplotlib.pyplot as plt
import pandas as pd

from bem.config import INTERIM_DIR, PROCESSED_DIR
from bem.geo.districts import district_colors, district_names

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

places = pd.read_csv(INTERIM_DIR / "places.csv")
texts = pd.read_csv(INTERIM_DIR / "texts_topics.csv", parse_dates=["created_utc"])

print(f"заведений: {len(places)}, текстов: {len(texts)}")

## 1. Заведения: что мы вообще собрали

Первое, что делаешь с новым датасетом, - смотришь на его форму и пропуски.
Не на средние и не на графики: сначала надо понять, что перед тобой.

In [ ]:
places.info()
print()
print("Пропуски по колонкам:")
print((places.isna().mean() * 100).round(1).sort_values(ascending=False).to_string())

**Что здесь видно.** `cuisine`, `opening_hours` и `website` заполнены
меньше чем наполовину. Это нормально для OpenStreetMap: его наполняют
волонтёры, и они заполняют то, что им интересно. Строить фичи на колонке
с 80% пропусков нельзя - это будет модель волонтёрской активности,
а не города.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

counts = places["district"].value_counts()
colors = [district_colors().get(k, "#999") for k in counts.index]
axes[0].barh([district_names().get(k, k) for k in counts.index], counts.values, color=colors)
axes[0].set_title("Заведений по районам")
axes[0].invert_yaxis()

cat = places["category"].value_counts()
axes[1].barh(cat.index, cat.values, color="#5c6bc0")
axes[1].set_title("Заведений по категориям")
axes[1].invert_yaxis()

plt.tight_layout()

## 2. Тексты: объём и распределение во времени

Ключевой вопрос - хватит ли данных, когда мы порежем их по районам
И по месяцам одновременно. Обычно оказывается, что нет.

In [ ]:
texts["month"] = texts["created_utc"].dt.to_period("M")
cell_sizes = texts.groupby(["district", "month"]).size()

print("Текстов в ячейке (район x месяц):")
print(cell_sizes.describe().round(1).to_string())
print()
print(f"ячеек, где меньше 20 текстов: {(cell_sizes < 20).mean():.0%}")

cell_sizes.hist(bins=30, figsize=(9, 3.5), color="#5c6bc0")
plt.axvline(20, color="red", linestyle="--", label="порог надёжности n=20")
plt.title("Сколько текстов приходится на одну ячейку район x месяц")
plt.legend();

**Вывод, который надо сделать здесь, а не после построения красивых
графиков:** значительная часть ячеек мельче порога надёжности. Значит,
на всех помесячных графиках обязательны доверительные интервалы, а точки
с малым n надо помечать. Это решение принято ЗДЕСЬ и реализовано
в `src/bem/analysis/timeseries.py`.

In [ ]:
by_month = texts.groupby(texts["created_utc"].dt.to_period("M")).size()
by_month.plot(figsize=(13, 3.5), color="#5c6bc0", linewidth=2)
plt.title("Число текстов по месяцам")
plt.ylabel("текстов")
plt.grid(alpha=0.3);

## 3. Согласие моделей между собой

Полезный приём: посмотреть, где VADER и DistilBERT расходятся.
Тексты, на которых модели спорят, - самые интересные. Обычно именно там
прячутся сарказм, смешанные оценки и просто сложные формулировки.

In [ ]:
agreement = pd.crosstab(texts["vader_label"], texts["distilbert_label"])
print("Согласие моделей (строки - VADER, столбцы - DistilBERT):")
print(agreement.to_string())

same = (texts["vader_label"] == texts["distilbert_label"]).mean()
print(f"\nмодели согласны в {same:.1%} случаев")

In [ ]:
# Смотрим глазами на тексты, где модели радикально разошлись.
# Ручная проверка выборки - обязательный этап. Метрика может быть
# отличной при том, что модель систематически ошибается на важном классе.
disagree = texts[
    ((texts.vader_label == "pos") & (texts.distilbert_label == "neg"))
    | ((texts.vader_label == "neg") & (texts.distilbert_label == "pos"))
]
print(f"радикальных расхождений: {len(disagree)}\n")

for _, r in disagree.head(8).iterrows():
    print(f"VADER={r.vader_label} / DistilBERT={r.distilbert_label}")
    print(f"  {r.text[:160]}\n")

## 4. Проверка на утечку: не предсказываем ли мы район по названию района

Тексты содержат названия районов. Если модель тем ухватится за них,
она будет кластеризовать по географии, а не по смыслу - а географию
мы и так знаем.

Именно эта проблема всплыла при первом прогоне BERTopic: тема
«north laine» вобрала 71% текстов района. Лечится стоп-словами
(`DISTRICT_STOPWORDS` в `src/bem/nlp/topics.py`).

In [ ]:
district_words = ["north laine", "kemptown", "the lanes", "seafront", "hove"]
for word in district_words:
    share = texts["text"].str.contains(word, case=False, na=False).mean()
    print(f"{word:<14} встречается в {share:6.1%} текстов")

## 5. Что дальше

Всё, что найдено здесь, уже перенесено в код:

* пропуски в OSM-полях → не используем `cuisine` и `opening_hours` как признаки;
* маленькие ячейки район×месяц → доверительные интервалы в `analysis/timeseries.py`;
* названия районов в текстах → стоп-слова в `nlp/topics.py`;
* расхождения моделей → сравнение в `scripts/03_sentiment.py`.

Ноутбук больше не нужен для работы пайплайна - он остаётся как
документ о том, КАК принимались решения.